In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
import joblib

In [17]:
# CARREGAMENTO DOS DADOS
df = pd.read_csv('Obesity.csv')

In [18]:
# ANÁLISE EXPLORATÓRIA
print("=== Estrutura Geral da Base ===")
print(df.info())

=== Estrutura Geral da Base ===
<class 'pandas.DataFrame'>
RangeIndex: 2111 entries, 0 to 2110
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Gender          2111 non-null   str    
 1   Age             2111 non-null   float64
 2   Height          2111 non-null   float64
 3   Weight          2111 non-null   float64
 4   family_history  2111 non-null   str    
 5   FAVC            2111 non-null   str    
 6   FCVC            2111 non-null   float64
 7   NCP             2111 non-null   float64
 8   CAEC            2111 non-null   str    
 9   SMOKE           2111 non-null   str    
 10  CH2O            2111 non-null   float64
 11  SCC             2111 non-null   str    
 12  FAF             2111 non-null   float64
 13  TUE             2111 non-null   float64
 14  CALC            2111 non-null   str    
 15  MTRANS          2111 non-null   str    
 16  Obesity         2111 non-null   str    
dtypes: float64(8

In [19]:
print("\n=== Estatísticas Descritivas ===")
print(df.describe())


=== Estatísticas Descritivas ===
               Age       Height       Weight         FCVC          NCP  \
count  2111.000000  2111.000000  2111.000000  2111.000000  2111.000000   
mean     24.312600     1.701677    86.586058     2.419043     2.685628   
std       6.345968     0.093305    26.191172     0.533927     0.778039   
min      14.000000     1.450000    39.000000     1.000000     1.000000   
25%      19.947192     1.630000    65.473343     2.000000     2.658738   
50%      22.777890     1.700499    83.000000     2.385502     3.000000   
75%      26.000000     1.768464   107.430682     3.000000     3.000000   
max      61.000000     1.980000   173.000000     3.000000     4.000000   

              CH2O          FAF          TUE  
count  2111.000000  2111.000000  2111.000000  
mean      2.008011     1.010298     0.657866  
std       0.612953     0.850592     0.608927  
min       1.000000     0.000000     0.000000  
25%       1.584812     0.124505     0.000000  
50%       2.00000

In [20]:
print("\n=== Distribuição da Variável Alvo ===")
print(df['Obesity'].value_counts())


=== Distribuição da Variável Alvo ===
Obesity
Obesity_Type_I         351
Obesity_Type_III       324
Obesity_Type_II        297
Overweight_Level_I     290
Overweight_Level_II    290
Normal_Weight          287
Insufficient_Weight    272
Name: count, dtype: int64


In [21]:
# TRATAMENTO E LIMPEZA DOS DADOS 
colunas_para_arredondar = ['FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
for col in colunas_para_arredondar:
    if col in df.columns:
        df[col] = df[col].round().astype(int)

In [22]:
# DIVISÃO EM TREINO E TESTE
target_col = 'Obesity'
X_raw = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

# CLASSES CUSTOMIZADAS PARA A PIPELINE
class RemoverAlturaPeso(BaseEstimator, TransformerMixin):
    def __init__(self, features_to_drop=['Weight', 'Height']):
        self.features_to_drop = features_to_drop
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        X_copy = X.copy()
        cols_present = [col for col in self.features_to_drop if col in X_copy.columns]
        return X_copy.drop(columns=cols_present)


class CustomLabelEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.binary_mappings = {
            'Gender': {'Masculino': 0, 'Feminino': 1, 'Male': 0, 'Female': 1},
            'family_history': {'Não': 0, 'Sim': 1, 'no': 0, 'yes': 1},
            'FAVC': {'Não': 0, 'Sim': 1, 'no': 0, 'yes': 1},
            'SMOKE': {'Não': 0, 'Sim': 1, 'no': 0, 'yes': 1},
            'SCC': {'Não': 0, 'Sim': 1, 'no': 0, 'yes': 1}
        }
        self.ordinal_scale = {
            'Não': 0, 'Às vezes': 1, 'Frequentemente': 2, 'Sempre': 3,
            'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3
        }
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        X_copy = X.copy()
        
        for col, mapping in self.binary_mappings.items():
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].map(mapping).fillna(0).astype(int)
                
        for col in ['CAEC', 'CALC']:
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].map(self.ordinal_scale).fillna(0).astype(int)
                
        return X_copy


class CustomOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.categories = ['Public_Transportation', 'Walking', 'Automobile', 'Motorbike', 'Bike']
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        X_copy = X.copy()
        
        if 'MTRANS' in X_copy.columns:
            for cat in self.categories[1:]:
                col_name = f'MTRANS_{cat}'
                X_copy[col_name] = (X_copy['MTRANS'] == cat).astype(int)
            X_copy = X_copy.drop(columns=['MTRANS'])
            
        return X_copy

# CONSTRUÇÃO E TREINAMENTO DA PIPELINE
pipeline_obesidade = Pipeline([
    ('remover_altura_peso', RemoverAlturaPeso()),
    ('custom_label_encoder', CustomLabelEncoder()),
    ('one_hot_encoder', CustomOneHotEncoder()),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
])

pipeline_obesidade.fit(X_train, y_train)

# VALIDAÇÃO E MÉTRICAS
y_pred = pipeline_obesidade.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("\n=== Desempenho Final ===")
print(f"Acurácia Global: {accuracy:.4f}\n")
print(classification_report(y_test, y_pred))

# SALVAR
joblib.dump(pipeline_obesidade, 'modelo_obesidade.pkl')
print("\nArquivo 'modelo_obesidade.pkl' gerado.")


=== Desempenho Final ===
Acurácia Global: 0.8061

                     precision    recall  f1-score   support

Insufficient_Weight       0.83      0.89      0.86        54
      Normal_Weight       0.62      0.66      0.64        58
     Obesity_Type_I       0.77      0.81      0.79        70
    Obesity_Type_II       0.96      0.87      0.91        60
   Obesity_Type_III       0.97      0.98      0.98        65
 Overweight_Level_I       0.80      0.69      0.74        58
Overweight_Level_II       0.70      0.72      0.71        58

           accuracy                           0.81       423
          macro avg       0.81      0.80      0.80       423
       weighted avg       0.81      0.81      0.81       423


Arquivo 'modelo_obesidade.pkl' gerado.
